In [1]:
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

from catboost import CatBoostClassifier

from pathlib import Path
import shutil
import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

import dice_ml

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
np.random.seed(0)
N = 600
outcome = np.random.choice([0, 1], size=N, p=[0.62, 0.38])

demo_df = pd.DataFrame({
    "age":        np.where(outcome == 1,
                            np.random.normal(55, 12, N),
                            np.random.normal(45, 13, N)).clip(18, 90),
    "bmi":        np.where(outcome == 1,
                            np.random.normal(28.5, 5, N),
                            np.random.normal(25.0, 4, N)).clip(15, 55),
    "systolic_bp":np.where(outcome == 1,
                            np.random.normal(145, 20, N),
                            np.random.normal(125, 15, N)).clip(80, 220),
    "glucose":    np.where(outcome == 1,
                            np.random.gamma(shape=3, scale=40, size=N) + 80,
                            np.random.gamma(shape=2.5, scale=30, size=N) + 70),
    "cholesterol":np.random.normal(200, 40, N).clip(100, 350),
    "creatinine": np.abs(np.random.normal(1.1, 0.4, N)),
    "sex":        np.random.choice([0, 1], N),
    "smoker":     np.where(outcome == 1,
                            np.random.choice([0, 1], N, p=[0.45, 0.55]),
                            np.random.choice([0, 1], N, p=[0.72, 0.28])),
    "outcome":    outcome,
})
demo_df.head()

,age,bmi,systolic_bp,glucose,cholesterol,creatinine,sex,smoker,outcome
0,47.728,27.027,123.690,189.779,286.638,1.077,0,0,0
1,43.254,30.036,191.748,135.965,180.150,1.375,1,1,1
2,47.892,18.909,80.000,127.061,243.408,0.683,0,0,0
3,43.851,27.228,124.293,154.772,228.738,1.219,0,0,0
4,46.279,17.647,137.478,127.206,202.024,2.280,1,0,0


In [19]:
from module.utils2 import eda

# Introduce ~4 % missing values
for col in ["bmi", "glucose", "creatinine"]:
    idx = np.random.choice(demo_df.index, size=int(0.04 * N), replace=False)
    demo_df.loc[idx, col] = np.nan

eda.run_eda(demo_df, target="outcome", output_dir=Path("eda_output_demo"))


  Scientific EDA  |  n=600  |  target='outcome'

Numerical features (8): ['age', 'bmi', 'systolic_bp', 'glucose', 'cholesterol', 'creatinine', 'sex', 'smoker']
Binary categorical features (0): []

[Tables]
  Table 1 saved → eda_output_demo\table1_descriptive.csv
  Table 1 LaTeX → eda_output_demo\table1_descriptive.tex

[Figures]
  Saved → eda_output_demo\fig1_distributions.png
  Saved → eda_output_demo\fig2_boxplots.png
  Saved → eda_output_demo\fig12_histogram.png
  Saved → eda_output_demo\fig5_missing_values.png

  EDA complete. All outputs in: eda_output_demo/



## Read Config File

In [4]:
config_path = Path(r'experiments')
config_filename =  "bin_cf_final.yml"
config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - Counterfactual Analysis (Final)',
  'classification_type': 'binary',
  'stage': 'counterfactuals',
  'tag': 'final',
  'verbosity': 0,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'code': 'catboost', 'name': 'CatBoost'},
 'explainability': {'ksplit_trained_model_results_file': 'binary\\explainability\\catboost\\final\\catboost_ksplit_trained_models.joblib',
  'rundate': '2026-03-18',
  'tag': 'final'},
 'dice': {'method': 'genetic',
  'global_cf': {'total_CFs': 20, 'posthoc_sparsity_algorithm': 'binary'},
  'local_cf': {'nrepeats': 3,
   'total_CFs': 20,
   'posthoc_sparsity_algorithm': 'binary',
   'posthoc_sparsity_param': 0.05,
   'proximity_weight': 0.5,
   'diversity_weight': 1.0,
   'categorical_penalty': 0.1,
   'algorithm': 'DiverseCF'},
  'sufficiency': {'maxiterations': 200},
  'necessity': {'maxiterations': 500, 'total_CFs': 2, 'nrepeats': 5}},
 'reporting':

#### Set output directory

In [5]:
outputdir = config_path /  config.experiment.classification_type /  'eda'
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments\binary\eda


## Data Loading

In [11]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
target_col = "Confirmed_Binary_DPN"
y = dfdpn[target_col]
X.shape, y.shape

((190, 40), (190,))

In [7]:
dfXy = pd.concat([X, y], axis=1)
X.shape, y.shape, dfXy.shape

((190, 40), (190,), (190, 41))

In [8]:
dfXy.head(2)

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,1,64.0,1,7.0,1.0,15.0,0,0,0,0,0,1,1,1,1,9.0,0.00,0.0,0.00,0.0,0.0,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.0,20.7,10.35,0.03,0.02,0.0,12.0,0.0,33.0,13.0,42.0,34.0,1
1,0,59.0,1,1.0,0.0,5.6,1,0,0,0,0,0,0,0,0,4.0,19.41,52.3,14.21,61.9,49.3,3.55,14.34,10.55,42.5,19.54,55.7,15.09,61.2,48.3,3.30,13.29,9.48,43.3,39.0,5.0,38.0,28.0,50.0,39.0,0


## EDA

In [35]:
from module.utils2 import eda
cols = ['SEX', 'SUBJ', 'INSULIN'] + D.comorbidity_cols + D.neuro_cols + [target_col]
eda.run_eda(dfXy[cols], 
            target=target_col, 
            output_dir=outputdir/'categorical',
            title="Binary Data from Profile, Commorbidity, Neurological Study",
            grid_shape=(4,3),
            continuous=False,
            )

cols = ['AGE', 'DM_DUR', 'HBA1C', 'MNSI'] + [target_col]
eda.run_eda(dfXy[cols], 
            target=target_col, 
            output_dir=outputdir/'profile',
            title="Continuous Data from Profile and MNSI Data",
            grid_shape=(2,2),
            continuous=True,
            )

cols = D.ncs_cols + [target_col]
eda.run_eda(dfXy[cols], 
            target=target_col, 
            title="Nerve Conduction Studies",
            grid_shape=(5,4),
            output_dir=outputdir/'ncs',
            continuous=True
            )

cols = D.sudo_cols + [target_col]
eda.run_eda(dfXy[cols], 
            target=target_col, 
            title="Sudoscan",
            grid_shape=(2,3),
            output_dir=outputdir/'sudo',
            continuous=True
            )


  Scientific EDA  |  n=190  |  target='Confirmed_Binary_DPN'

Numerical features (12): ['SEX', 'SUBJ', 'INSULIN', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR']
Binary categorical features (0): []

[Tables]
  Table 1 saved → experiments\binary\eda\categorical\table1_descriptive.csv
  Table 1 LaTeX → experiments\binary\eda\categorical\table1_descriptive.tex

[Figures]
  Saved → experiments\binary\eda\categorical\fig12_histogram.png
  Saved → experiments\binary\eda\categorical\fig5_missing_values.png

  EDA complete. All outputs in: experiments\binary\eda\categorical/


  Scientific EDA  |  n=190  |  target='Confirmed_Binary_DPN'

Numerical features (4): ['AGE', 'DM_DUR', 'HBA1C', 'MNSI']
Binary categorical features (0): []

[Tables]
  Table 1 saved → experiments\binary\eda\profile\table1_descriptive.csv
  Table 1 LaTeX → experiments\binary\eda\profile\table1_descriptive.tex

[Figures]
  Saved → experiments\binary\eda\profile\fig1_distributions.png
  